<p align="center">
  <a href="https://github.com/wavekat/wavekat-lab">
    <img src="https://github.com/wavekat/wavekat-brand/raw/main/assets/banners/wavekat-lab-narrow.svg" alt="WaveKat Lab">
  </a>
</p>

# Smart-Turn — train

Fine-tune `openai/whisper-tiny`'s encoder with an attention-pooling
classification head on the smart-turn dataset produced by
[`wk exports adapt smart-turn`](https://github.com/wavekat/wavekat-cli).

The pipeline mirrors [pipecat-ai/smart-turn](https://github.com/pipecat-ai/smart-turn)'s
`train.py`, trimmed down for the smaller ad-hoc snapshots this lab tends
to produce:

1. Reuse the loader from `01_load_export.ipynb` to pick up Parquet shards.
2. Define the Whisper-encoder + attention-pool + classifier model.
3. Wrap each split in an on-the-fly mel-spectrogram dataset.
4. Train with HF `Trainer`, validating once per epoch.
5. Save the HF checkpoint into `CHECKPOINT_DIR` so `03_eval.ipynb` can pick it up.

**Compute.** CUDA is the happy path. Apple Silicon MPS works as a
slower alternative (fp32 only, expect occasional silent CPU fallbacks)
— enough to iterate on the pipeline locally before pushing the real
run to a GPU box. On pure CPU the training step is too slow to finish.
With a small CUDA GPU (T4, A10) the run completes in under an hour for
~1k samples.

## Configure

`EXPORT_DIR` matches the convention from `01_load_export.ipynb` —
override it if your `wk exports adapt smart-turn` snapshot landed
somewhere else. `CHECKPOINT_DIR` is where the HF checkpoint and any
intermediate `Trainer` state live; `03_eval.ipynb` reads from the same
path.

In [ ]:
import os
from pathlib import Path

EXPORT_DIR = Path("../../datasets/smart-turn-zh").resolve()
CHECKPOINT_DIR = Path("../../checkpoints/smart-turn-zh").resolve()
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "openai/whisper-tiny"
TARGET_SR = 16_000
CHUNK_LENGTH = 8  # seconds — model input cap, matches upstream

# Hyperparameters — tuned for ~1k-sample exports. Upstream defaults
# (batch 384, 4 epochs) assume 270k samples; with ~1k we drop the batch
# and run more epochs so each sample is seen enough times.
BATCH_SIZE = 16
GRAD_ACCUM = 1
EVAL_BATCH_SIZE = 32
EPOCHS = 8
LR = 5e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

assert EXPORT_DIR.exists(), (
    f"{EXPORT_DIR.name} not found. Re-run `wk exports adapt smart-turn --out {EXPORT_DIR}` "
    f"or override EXPORT_DIR above."
)
print("export dir   :", EXPORT_DIR.name)
print("checkpoint   :", CHECKPOINT_DIR.name)
print("base model   :", BASE_MODEL)
print("epochs/batch :", f"{EPOCHS} / {BATCH_SIZE} (x {GRAD_ACCUM} accum)")
print("✅ config ready")

## Load dataset

Same loader as `01_load_export.ipynb`. Keeping the block stable means a
new export drops in without touching the training notebook.

In [ ]:
from datasets import load_dataset, Audio, disable_progress_bars

disable_progress_bars()

data_files = {
    split: str(EXPORT_DIR / f"{split}.parquet")
    for split in ("train", "validation", "test")
    if (EXPORT_DIR / f"{split}.parquet").exists()
}
assert "train" in data_files, "train.parquet missing — adapt step did not produce it."

ds = load_dataset("parquet", data_files=data_files)
ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))
print(ds)
print("✅ dataset loaded")

## Model

Whisper Tiny encoder → attention pooling over time → 2-layer classifier
head producing a single logit. The encoder is initialised from the
`openai/whisper-tiny` weights; the head is fresh.

`max_source_positions = 400` matches the 8 s × 16 kHz / 320 stride / 2
the encoder produces — it has to be set before the encoder is built so
positional embeddings are sized correctly.

In [ ]:
import torch
import torch.nn as nn
from transformers import WhisperConfig, WhisperPreTrainedModel
from transformers.models.whisper.modeling_whisper import WhisperEncoder


class SmartTurnModel(WhisperPreTrainedModel):
    """Whisper encoder + attention pooling + binary classifier."""

    def __init__(self, config: WhisperConfig):
        super().__init__(config)
        config.max_source_positions = 400
        self.encoder = WhisperEncoder(config)

        hidden = config.d_model

        self.pool_attention = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.Tanh(),
            nn.Linear(256, 1),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

        self.post_init()

    def forward(self, input_features, labels=None):
        enc = self.encoder(input_features).last_hidden_state          # (B, T, H)
        attn = torch.softmax(self.pool_attention(enc).squeeze(-1), -1)
        pooled = torch.bmm(attn.unsqueeze(1), enc).squeeze(1)         # (B, H)

        logits = self.classifier(pooled).squeeze(-1)                  # (B,)
        probs = torch.sigmoid(logits)

        loss = None
        if labels is not None:
            pos_weight = (
                ((labels == 0).sum() / (labels == 1).sum().clamp(min=1))
                .clamp(0.1, 10.0)
            )
            loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)(logits, labels.float())

        return {"loss": loss, "logits": probs}


print("✅ SmartTurnModel defined")

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")  # Apple Silicon — fp32 only, expect occasional silent CPU fallbacks
else:
    device = torch.device("cpu")
print("device:", device)
if device.type == "cuda":
    print("gpu   :", torch.cuda.get_device_name(0))
    print("vram  :", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

config = WhisperConfig.from_pretrained(BASE_MODEL)
model = SmartTurnModel(config)

# Load pretrained encoder weights, leave head freshly initialised.
pretrained = SmartTurnModel.from_pretrained(BASE_MODEL, config=config, ignore_mismatched_sizes=True)
model.encoder.load_state_dict(pretrained.encoder.state_dict())
del pretrained

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params: {total:,} total, {trainable:,} trainable")
print("✅ model initialised")

## Feature extraction

Mel spectrograms are computed on the fly inside the dataset rather than
materialised up-front. With ~1k clips this would fit in RAM either way,
but matching the upstream layout means we can flip to a larger snapshot
without rewriting this block.

`truncate_to_last_n_seconds` keeps the trailing audio (where the
end-of-turn signal lives) and zero-pads the start if a clip is shorter
than the 8 s window.

In [ ]:
import numpy as np
from torch.utils.data import Dataset
from transformers import WhisperFeatureExtractor


def truncate_to_last_n_seconds(audio_array: np.ndarray, sr: int, n: int = CHUNK_LENGTH) -> np.ndarray:
    max_samples = sr * n
    if len(audio_array) > max_samples:
        return audio_array[-max_samples:]
    if len(audio_array) < max_samples:
        pad = np.zeros(max_samples - len(audio_array), dtype=audio_array.dtype)
        return np.concatenate([pad, audio_array])
    return audio_array


class SmartTurnDataset(Dataset):
    """Wraps a HF dataset, extracting mel features on the fly."""

    def __init__(self, hf_dataset, feature_extractor):
        self.ds = hf_dataset
        self.fe = feature_extractor

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]
        audio = sample["audio"]
        arr = np.array(audio["array"], dtype=np.float32)
        arr = truncate_to_last_n_seconds(arr, audio["sampling_rate"])

        features = self.fe(
            arr,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding="max_length",
            max_length=CHUNK_LENGTH * TARGET_SR,
            truncation=True,
            do_normalize=True,
        )
        return {
            "input_features": features["input_features"].squeeze(0),  # (80, 800)
            "labels": torch.tensor(float(sample["endpoint_bool"])),
        }


feature_extractor = WhisperFeatureExtractor(chunk_length=CHUNK_LENGTH)

train_dataset = SmartTurnDataset(ds["train"], feature_extractor)
eval_dataset = SmartTurnDataset(ds.get("validation", ds["train"]), feature_extractor)

print(f"train: {len(train_dataset):,}")
print(f"eval : {len(eval_dataset):,}")

# Sanity-check one sample so a downstream shape mismatch fails here, not 30 min into training.
sample0 = train_dataset[0]
print("sample input_features:", tuple(sample0["input_features"].shape))
print("sample label         :", float(sample0["labels"]))
print("✅ feature extractor + datasets ready")

## Train

Eval/save per epoch — with ~60 train steps per epoch the per-step
strategy is too noisy to drive `load_best_model_at_end`. `f1` is the
selection metric since label balance is uneven across snapshots.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import Trainer, TrainingArguments


def compute_metrics(eval_pred):
    probs, labels = eval_pred
    preds = (probs > 0.5).astype(int).flatten()
    labels = labels.astype(int).flatten()
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }


use_cuda = torch.cuda.is_available()
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    run_name="smart-turn-zh",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    dataloader_num_workers=2,
    bf16=use_cuda and torch.cuda.is_bf16_supported(),
    fp16=use_cuda and not torch.cuda.is_bf16_supported(),
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb" if os.environ.get("WANDB_API_KEY") else "none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

print("effective batch:", BATCH_SIZE * GRAD_ACCUM)
print("precision      :", "bf16" if training_args.bf16 else "fp16" if training_args.fp16 else "fp32")
print("wandb          :", "on" if training_args.report_to == ["wandb"] else "off")
print("✅ trainer ready")

In [ ]:
trainer.train()
print("✅ training complete")

## Save

`03_eval.ipynb` reloads from this path with `SmartTurnModel.from_pretrained`.
The feature extractor goes alongside so inference doesn't have to
re-derive the mel config.

In [ ]:
model.save_pretrained(CHECKPOINT_DIR)
feature_extractor.save_pretrained(CHECKPOINT_DIR)
print("saved:", CHECKPOINT_DIR.name)
print("files:", sorted(p.name for p in CHECKPOINT_DIR.iterdir() if p.is_file()))
print("✅ checkpoint saved")

## Next

`03_eval.ipynb` reloads this checkpoint and runs:

- held-out metrics on `ds["test"]`,
- ONNX FP32 export + INT8 static quantization,
- a CPU-side latency benchmark so you can decide which artifact ships.